# Topic 3: Built-in Functions

> **Phase 3 → Week 4 → Topic 3**
>
> Prerequisites: Topic 1 & 2

---

## Why Built-in Functions Over Python UDFs

```
Python UDF:              Built-in Function:
@udf                     F.upper(col)
def my_upper(s):    vs
    return s.upper()

- Python process        - Runs in JVM (Tungsten)
- Serialization cost    - No serialization
- Catalyst can't        - Catalyst optimizes
  optimize              - 10-100x faster
- Slower                - Always prefer these
```

**Rule: Always use `pyspark.sql.functions` over Python UDFs whenever possible.**

---

## Interview Cheat Sheet

**Q: Why are Spark built-in functions faster than Python UDFs?**
> Built-in functions execute natively in the JVM via Tungsten — no Python-JVM serialization, and Catalyst can inspect and optimize them. Python UDFs require serializing each row from JVM to Python process, executing, then serializing back — row by row overhead adds up to 10-100x slower.

**Q: What's the difference between when() and coalesce() for conditional logic?**
> `when(condition, value).otherwise(default)` is general conditional logic — like an IF/ELSE. `coalesce(col1, col2, ...)` returns the FIRST non-null value from its arguments — it's specifically for null fallback/default values, not general conditions.

**Q: How do you handle date arithmetic in Spark?**
> Use `F.datediff(end, start)` for days between dates, `F.months_between(end, start)` for months, `F.date_add(col, n)` / `F.date_sub(col, n)` to add/subtract days. Always use `F.to_date(col, format)` or `F.to_timestamp()` to parse string dates first.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Week4 - Built-in Functions") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

customers = spark.read.csv("/workspace/week4/data/customers.csv", header=True, inferSchema=True)
orders    = spark.read.csv("/workspace/week4/data/orders.csv",    header=True, inferSchema=True)
print("Data loaded")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/14 03:34:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Data loaded


## Part 1: String Functions

In [2]:
string_df = spark.createDataFrame([
    ("Alice Sharma",   "alice@example.com",   "  hello world  "),
    ("Bob Chen",       "bob.chen@gmail.com",   "  SPARK is FUN  "),
    ("Carol Williams", "carol@company.co.uk",  "data engineering"),
], ["name", "email", "notes"])

string_ops = string_df.select(
    "name",
    F.upper("name").alias("upper_name"),
    F.lower("name").alias("lower_name"),
    F.length("name").alias("name_len"),
    F.trim("notes").alias("trimmed"),
    F.ltrim("notes").alias("ltrimmed"),
    F.split("name", " ")[0].alias("first_name"),
    F.split("name", " ")[1].alias("last_name"),
    F.substring("email", 1, 5).alias("email_prefix"),
    F.regexp_extract("email", r"@(.+)", 1).alias("email_domain"),
    F.regexp_replace("notes", "  ", " ").alias("single_space"),
)

print("String functions:")
string_ops.show(truncate=False)

String functions:


+--------------+--------------+--------------+--------+----------------+----------------+----------+---------+------------+-------------+----------------+
|name          |upper_name    |lower_name    |name_len|trimmed         |ltrimmed        |first_name|last_name|email_prefix|email_domain |single_space    |
+--------------+--------------+--------------+--------+----------------+----------------+----------+---------+------------+-------------+----------------+
|Alice Sharma  |ALICE SHARMA  |alice sharma  |12      |hello world     |hello world     |Alice     |Sharma   |alice       |example.com  | hello world    |
|Bob Chen      |BOB CHEN      |bob chen      |8       |SPARK is FUN    |SPARK is FUN    |Bob       |Chen     |bob.c       |gmail.com    | SPARK is FUN   |
|Carol Williams|CAROL WILLIAMS|carol williams|14      |data engineering|data engineering|Carol     |Williams |carol       |company.co.uk|data engineering|
+--------------+--------------+--------------+--------+---------------

In [3]:
# concat, concat_ws, format_string, lpad, rpad
concat_df = customers.select(
    F.concat(F.col("name"), F.lit(" ("), F.col("country"), F.lit(")")).alias("name_country"),
    F.concat_ws(" | ", "name", "city", "country").alias("pipe_separated"),
    F.format_string("Customer %s from %s", "name", "country").alias("formatted"),
    F.lpad(F.col("customer_id").cast("string"), 4, "0").alias("padded_id"),
    F.initcap(F.lower("country")).alias("title_case"),
)

print("Concat, format, pad functions:")
concat_df.show(truncate=False)

Concat, format, pad functions:


+------------------------+--------------------------------+------------------------------------+---------+-----------+
|name_country            |pipe_separated                  |formatted                           |padded_id|title_case |
+------------------------+--------------------------------+------------------------------------+---------+-----------+
|Alice Sharma (India)    |Alice Sharma | Mumbai | India   |Customer Alice Sharma from India    |0001     |India      |
|Bob Chen (China)        |Bob Chen | Beijing | China      |Customer Bob Chen from China        |0002     |China      |
|Carol Williams (USA)    |Carol Williams | New York | USA |Customer Carol Williams from USA    |0003     |Usa        |
|Dave Kumar (India)      |Dave Kumar | Bangalore | India  |Customer Dave Kumar from India      |0004     |India      |
|Eve Müller (Germany)    |Eve Müller | Berlin | Germany   |Customer Eve Müller from Germany    |0005     |Germany    |
|Frank Tanaka (Japan)    |Frank Tanaka | Tokyo |

In [4]:
# contains, startswith, endswith, like, rlike
filter_df = customers.filter(
    F.col("name").contains("a")   # name contains lowercase 'a'
).select("name", "country")

print("Names containing 'a':")
filter_df.show()

# Pattern matching
pattern_df = customers.select(
    "name",
    F.col("name").startswith("A").alias("starts_A"),
    F.col("name").endswith("n").alias("ends_n"),
    F.col("name").like("%ar%").alias("sql_like"),
    F.col("name").rlike("^[A-E]").alias("regex_A_to_E"),
)
print("Pattern matching:")
pattern_df.show()

Names containing 'a':


+--------------+-----------+
|          name|    country|
+--------------+-----------+
|  Alice Sharma|      India|
|Carol Williams|        USA|
|    Dave Kumar|      India|
|  Frank Tanaka|      Japan|
|    Grace Park|South Korea|
|  Henry Okafor|    Nigeria|
|   James Brown|         UK|
+--------------+-----------+

Pattern matching:


+--------------+--------+------+--------+------------+
|          name|starts_A|ends_n|sql_like|regex_A_to_E|
+--------------+--------+------+--------+------------+
|  Alice Sharma|    true| false|    true|        true|
|      Bob Chen|   false|  true|   false|        true|
|Carol Williams|   false| false|    true|        true|
|    Dave Kumar|   false| false|    true|        true|
|    Eve Müller|   false| false|   false|        true|
|  Frank Tanaka|   false| false|   false|       false|
|    Grace Park|   false| false|    true|       false|
|  Henry Okafor|   false| false|   false|       false|
|   Irene Rossi|   false| false|   false|       false|
|   James Brown|   false|  true|   false|       false|
+--------------+--------+------+--------+------------+



## Part 2: Date & Time Functions

In [5]:
date_df = orders.select("order_id", "order_date", "amount")

date_ops = date_df.withColumn("order_date", F.to_date("order_date", "yyyy-MM-dd")) \
                  .withColumn("year",        F.year("order_date")) \
                  .withColumn("month",       F.month("order_date")) \
                  .withColumn("day",         F.dayofmonth("order_date")) \
                  .withColumn("day_of_week", F.dayofweek("order_date")) \
                  .withColumn("week_of_yr",  F.weekofyear("order_date")) \
                  .withColumn("quarter",     F.quarter("order_date")) \
                  .withColumn("plus_30days", F.date_add("order_date", 30)) \
                  .withColumn("today",       F.current_date()) \
                  .withColumn("days_ago",    F.datediff(F.current_date(), "order_date"))

print("Date extraction and arithmetic:")
date_ops.show(truncate=False)

Date extraction and arithmetic:


+--------+----------+-------+----+-----+---+-----------+----------+-------+-----------+----------+--------+
|order_id|order_date|amount |year|month|day|day_of_week|week_of_yr|quarter|plus_30days|today     |days_ago|
+--------+----------+-------+----+-----+---+-----------+----------+-------+-----------+----------+--------+
|O001    |2023-06-01|1299.99|2023|6    |1  |5          |22        |2      |2023-07-01 |2026-05-14|1078    |
|O002    |2023-06-05|79.98  |2023|6    |5  |2          |23        |2      |2023-07-05 |2026-05-14|1074    |
|O003    |2023-06-10|89.97  |2023|6    |10 |7          |23        |2      |2023-07-10 |2026-05-14|1069    |
|O004    |2023-06-12|1299.99|2023|6    |12 |2          |24        |2      |2023-07-12 |2026-05-14|1067    |
|O005    |2023-06-12|599.98 |2023|6    |12 |2          |24        |2      |2023-07-12 |2026-05-14|1067    |
|O006    |2023-06-15|49.99  |2023|6    |15 |5          |24        |2      |2023-07-15 |2026-05-14|1064    |
|O007    |2023-06-18|199.99 

In [6]:
# Timestamp functions
ts_df = spark.createDataFrame([
    ("2023-06-15 14:30:00",),
    ("2023-06-16 09:15:30",),
    ("2023-12-31 23:59:59",),
], ["ts_string"])

ts_ops = ts_df.withColumn("ts",       F.to_timestamp("ts_string", "yyyy-MM-dd HH:mm:ss")) \
              .withColumn("hour",     F.hour("ts")) \
              .withColumn("minute",   F.minute("ts")) \
              .withColumn("date",     F.to_date("ts")) \
              .withColumn("unix_ts",  F.unix_timestamp("ts")) \
              .withColumn("truncate_hr", F.date_trunc("hour", "ts"))  # round down to hour

print("Timestamp functions:")
ts_ops.show(truncate=False)

Timestamp functions:


+-------------------+-------------------+----+------+----------+----------+-------------------+
|ts_string          |ts                 |hour|minute|date      |unix_ts   |truncate_hr        |
+-------------------+-------------------+----+------+----------+----------+-------------------+
|2023-06-15 14:30:00|2023-06-15 14:30:00|14  |30    |2023-06-15|1686839400|2023-06-15 14:00:00|
|2023-06-16 09:15:30|2023-06-16 09:15:30|9   |15    |2023-06-16|1686906930|2023-06-16 09:00:00|
|2023-12-31 23:59:59|2023-12-31 23:59:59|23  |59    |2023-12-31|1704067199|2023-12-31 23:00:00|
+-------------------+-------------------+----+------+----------+----------+-------------------+



In [7]:
# months_between + date formatting
signup_analysis = customers.withColumn(
    "signup_date", F.to_date("signup_date", "yyyy-MM-dd")
).withColumn(
    "months_since_signup", F.round(F.months_between(F.current_date(), "signup_date"), 1)
).withColumn(
    "signup_formatted",    F.date_format("signup_date", "dd MMM yyyy")
).select("name", "signup_date", "signup_formatted", "months_since_signup")

print("Months since signup:")
signup_analysis.show()

Months since signup:


+--------------+-----------+----------------+-------------------+
|          name|signup_date|signup_formatted|months_since_signup|
+--------------+-----------+----------------+-------------------+
|  Alice Sharma| 2022-01-15|     15 Jan 2022|               52.0|
|      Bob Chen| 2022-03-20|     20 Mar 2022|               49.8|
|Carol Williams| 2021-11-05|     05 Nov 2021|               54.3|
|    Dave Kumar| 2023-02-10|     10 Feb 2023|               39.1|
|    Eve Müller| 2022-07-18|     18 Jul 2022|               45.9|
|  Frank Tanaka| 2021-09-01|     01 Sep 2021|               56.4|
|    Grace Park| 2023-05-12|     12 May 2023|               36.1|
|  Henry Okafor| 2022-12-30|     30 Dec 2022|               40.5|
|   Irene Rossi| 2023-01-08|     08 Jan 2023|               40.2|
|   James Brown| 2021-06-22|     22 Jun 2021|               58.7|
+--------------+-----------+----------------+-------------------+



## Part 3: Math Functions

In [8]:
math_df = spark.createDataFrame([
    (1,  -15.7,  2.5,  100.0),
    (2,   23.4,  0.0,  200.0),
    (3,  -0.3,   9.0,   50.0),
    (4,   87.6, -3.0,  150.0),
], ["id", "val", "divisor", "price"])

math_ops = math_df.select(
    "id",
    F.abs("val").alias("abs_val"),
    F.round("val", 1).alias("rounded_1dp"),
    F.ceil("val").alias("ceiling"),
    F.floor("val").alias("floor"),
    F.sqrt("price").alias("sqrt_price"),
    F.pow("price", 2).alias("price_squared"),
    F.log("price").alias("log_price"),
    F.log(10.0, "price").alias("log10_price"),
    F.when(F.col("divisor") != 0, F.col("price") / F.col("divisor")).alias("safe_divide"),
)

print("Math functions:")
math_ops.show()

Math functions:


+---+-------+-----------+-------+-----+------------------+-------------+------------------+------------------+-----------------+
| id|abs_val|rounded_1dp|ceiling|floor|        sqrt_price|price_squared|         log_price|       log10_price|      safe_divide|
+---+-------+-----------+-------+-----+------------------+-------------+------------------+------------------+-----------------+
|  1|   15.7|      -15.7|    -15|  -16|              10.0|      10000.0| 4.605170185988092|               2.0|             40.0|
|  2|   23.4|       23.4|     24|   23|14.142135623730951|      40000.0| 5.298317366548036| 2.301029995663981|             NULL|
|  3|    0.3|       -0.3|      0|   -1|7.0710678118654755|       2500.0| 3.912023005428146|1.6989700043360185|5.555555555555555|
|  4|   87.6|       87.6|     88|   87| 12.24744871391589|      22500.0|5.0106352940962555| 2.176091259055681|            -50.0|
+---+-------+-----------+-------+-----+------------------+-------------+------------------+------

## Part 4: Conditional Functions — when(), otherwise(), coalesce()

In [9]:
# when().when().otherwise() — like IF/ELSEIF/ELSE or SQL CASE WHEN
orders_with_tier = orders.withColumn(
    "order_tier",
    F.when(F.col("amount") >= 1000, "High")
     .when(F.col("amount") >= 200,  "Medium")
     .when(F.col("amount") >= 50,   "Low")
     .otherwise("Micro")
)

print("Order tiers using when/otherwise:")
orders_with_tier.select("order_id", "amount", "order_tier").show()

print("Count per tier:")
orders_with_tier.groupBy("order_tier").count().show()

Order tiers using when/otherwise:


+--------+-------+----------+
|order_id| amount|order_tier|
+--------+-------+----------+
|    O001|1299.99|      High|
|    O002|  79.98|       Low|
|    O003|  89.97|       Low|
|    O004|1299.99|      High|
|    O005| 599.98|    Medium|
|    O006|  49.99|     Micro|
|    O007| 199.99|       Low|
|    O008| 449.99|    Medium|
|    O009|  44.99|     Micro|
|    O010| 149.95|       Low|
|    O011|  69.98|       Low|
|    O012|2599.98|      High|
|    O013| 199.99|       Low|
|    O014|  99.98|       Low|
|    O015| 119.97|       Low|
|    O016|1299.99|      High|
|    O017| 119.98|       Low|
|    O018|  44.99|     Micro|
|    O019| 299.99|    Medium|
|    O020|  59.98|       Low|
+--------+-------+----------+

Count per tier:


+----------+-----+
|order_tier|count|
+----------+-----+
|      High|    4|
|       Low|   10|
|     Micro|    3|
|    Medium|    3|
+----------+-----+



In [10]:
# coalesce() — first non-null from list of columns
# Great for providing fallback values across columns

fallback_df = spark.createDataFrame([
    (1, None,    None,    "default_A"),
    (2, "val_B", None,    "default_B"),
    (3, None,    "val_C2", "default_C"),
    (4, "val_D", "val_D2", "default_D"),
], ["id", "col1", "col2", "col3"])

fallback_df.withColumn(
    "first_non_null", F.coalesce("col1", "col2", "col3")
).show()

# nvl (alias for coalesce with two args) — SQL compatibility
spark_df = customers.withColumn(
    "tier_or_unknown", F.coalesce(F.col("tier"), F.lit("Unknown"))
)
print("Coalesce for null fallback:")
spark_df.select("name", "tier", "tier_or_unknown").show(5)

+---+-----+------+---------+--------------+
| id| col1|  col2|     col3|first_non_null|
+---+-----+------+---------+--------------+
|  1| NULL|  NULL|default_A|     default_A|
|  2|val_B|  NULL|default_B|         val_B|
|  3| NULL|val_C2|default_C|        val_C2|
|  4|val_D|val_D2|default_D|         val_D|
+---+-----+------+---------+--------------+

Coalesce for null fallback:


+--------------+------+---------------+
|          name|  tier|tier_or_unknown|
+--------------+------+---------------+
|  Alice Sharma|  Gold|           Gold|
|      Bob Chen|Silver|         Silver|
|Carol Williams|  Gold|           Gold|
|    Dave Kumar|Bronze|         Bronze|
|    Eve Müller|Silver|         Silver|
+--------------+------+---------------+
only showing top 5 rows



In [11]:
# Conditional aggregation — count/sum only rows meeting a condition
# Pattern: F.sum(F.when(condition, 1).otherwise(0))

order_summary = orders.groupBy("customer_id").agg(
    F.count("order_id").alias("total_orders"),
    F.sum(F.when(F.col("status") == "delivered", 1).otherwise(0)).alias("delivered_count"),
    F.sum(F.when(F.col("status") == "cancelled", 1).otherwise(0)).alias("cancelled_count"),
    F.sum(F.when(F.col("status") == "delivered", F.col("amount")).otherwise(0)).alias("delivered_revenue"),
    F.round(F.avg("amount"), 2).alias("avg_order_value")
)

print("Conditional aggregation (pivot-style):")
order_summary.show()

Conditional aggregation (pivot-style):


+-----------+------------+---------------+---------------+------------------+---------------+
|customer_id|total_orders|delivered_count|cancelled_count| delivered_revenue|avg_order_value|
+-----------+------------+---------------+---------------+------------------+---------------+
|          6|           2|              2|              0|            569.96|         284.98|
|          9|           1|              0|              1|               0.0|          69.98|
|          2|           2|              2|              0|            209.95|         104.98|
|          4|           2|              1|              0|             59.98|          54.99|
|          5|           2|              2|              0|244.98000000000002|         122.49|
|         10|           2|              1|              0|           2599.98|        1449.99|
|          1|           3|              3|              0|           1579.96|         526.65|
|          3|           3|              2|              0|  

## Part 5: Aggregate Functions Reference

In [12]:
# Common aggregate functions in one place
agg_demo = orders.agg(
    F.count("order_id").alias("total_count"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.sum("amount").alias("total_revenue"),
    F.avg("amount").alias("avg_order"),
    F.min("amount").alias("min_order"),
    F.max("amount").alias("max_order"),
    F.stddev("amount").alias("stddev_amount"),
    F.variance("amount").alias("variance"),
    F.first("order_date").alias("first_order_date"),
    F.last("order_date").alias("last_order_date"),
    F.collect_list("status").alias("all_statuses"),
    F.collect_set("status").alias("unique_statuses"),
)

print("Full aggregate summary of orders:")
agg_demo.show(truncate=False)

Full aggregate summary of orders:


+-----------+----------------+-------------+------------------+---------+---------+-----------------+-----------------+----------------+---------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------+
|total_count|unique_customers|total_revenue|avg_order         |min_order|max_order|stddev_amount    |variance         |first_order_date|last_order_date|all_statuses                                                                                                                                                                                                              |unique_statuses                            |
+-----------+----------------+-------------+------------------+---------+---------+-----------------+-----------------+----------------+---------------+----------------

## Exercises

1. From the `customers` table, extract the first name and last name into separate columns using `split()`.
2. For each order, calculate how many days ago it was placed (from today's date).
3. Using `when().otherwise()`, classify customers into: `New` (signup < 1 year ago), `Regular` (1-2 years), `Veteran` (2+ years).
4. Find the total amount for delivered orders vs non-delivered orders using conditional aggregation.
5. Build a full customer summary: name, total orders, total spend, avg order value, last order date.

In [13]:
# Exercise 1: Extract first/last name
customers.select(
    F.split("name", " ")[0].alias("first_name"),
    F.split("name", " ")[1].alias("last_name"),
    "country"
).show()

+----------+---------+-----------+
|first_name|last_name|    country|
+----------+---------+-----------+
|     Alice|   Sharma|      India|
|       Bob|     Chen|      China|
|     Carol| Williams|        USA|
|      Dave|    Kumar|      India|
|       Eve|   Müller|    Germany|
|     Frank|   Tanaka|      Japan|
|     Grace|     Park|South Korea|
|     Henry|   Okafor|    Nigeria|
|     Irene|    Rossi|      Italy|
|     James|    Brown|         UK|
+----------+---------+-----------+



In [14]:
# Exercise 3: Customer vintage classification
from pyspark.sql.window import Window

customers.withColumn(
    "signup_date", F.to_date("signup_date")
).withColumn(
    "months_ago", F.months_between(F.current_date(), "signup_date")
).withColumn(
    "segment",
    F.when(F.col("months_ago") < 12,  "New")
     .when(F.col("months_ago") < 24,  "Regular")
     .otherwise("Veteran")
).select("name", "signup_date", "months_ago", "segment").show()

+--------------+-----------+-----------+-------+
|          name|signup_date| months_ago|segment|
+--------------+-----------+-----------+-------+
|  Alice Sharma| 2022-01-15|51.96774194|Veteran|
|      Bob Chen| 2022-03-20|49.80645161|Veteran|
|Carol Williams| 2021-11-05|54.29032258|Veteran|
|    Dave Kumar| 2023-02-10|39.12903226|Veteran|
|    Eve Müller| 2022-07-18|45.87096774|Veteran|
|  Frank Tanaka| 2021-09-01|56.41935484|Veteran|
|    Grace Park| 2023-05-12|36.06451613|Veteran|
|  Henry Okafor| 2022-12-30|40.48387097|Veteran|
|   Irene Rossi| 2023-01-08|40.19354839|Veteran|
|   James Brown| 2021-06-22|58.74193548|Veteran|
+--------------+-----------+-----------+-------+

